# 통합 검색 P0 — 임베딩 모델 bake-off (ADR-0090 D5)

Colab **T4 GPU** 런타임에서 돈다(런타임 → 런타임 유형 변경 → T4). 후보 모델 × 차원(512·1024) × 텍스트 규칙(full·title) 을
판정 세트 `judgments.yml` 의 nDCG@10 으로 한 표에 낸다.

- **판정은 사람이 적는다.** `grade` 가 없는 질의는 평가에서 빠진다. 이 노트북은 정답을 만들지 않는다.
- 임베딩 텍스트 규칙·모델 스펙은 레포 `tools/embed` 의 것을 그대로 쓴다(두 구현 금지).
- Colab 은 클러스터에 닿지 않는다. 벡터는 parquet 으로 내리고, 업서트는 로컬의 `push --file`(P1) 이 한다.


In [ ]:
#@title 0. 설치 — 레포의 tools/embed 를 그대로 (규칙·모델 스펙의 유일한 구현)
%pip -q install "kgd-embed[model] @ git+https://github.com/1989v/msa.git#subdirectory=tools/embed"
%pip -q install bitsandbytes  # Qwen3-Embedding-8B 8bit 용 (CUDA)
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")


In [ ]:
#@title 1. 파라미터
PLACE_API = "https://api.1989v.com"          # 공개 GET /api/places/attractions
JUDGMENTS = "https://raw.githubusercontent.com/1989v/msa/main/docs/specs/2026-09-05-unified-search/judgments.yml"
LANG = "ko"                                    # 한국어 문서 45k. 영어 질의 6개는 en 코퍼스로 따로 돌린다(아래 4b)
DOC_LIMIT = None                               # 스모크 테스트면 300
MODELS = ["gemma-300m", "harrier-0.6b", "arctic-ko", "qwen3-4b", "qwen3-8b"]   # 무거운 순서가 뒤 — 앞에서 끊어도 표가 남는다
DIMS = [512, 1024]
RULES = ["full", "title"]
EXPORT_MODEL = None                            # 예: "arctic-ko" — 첫 채움 parquet 을 낼 모델(차원은 그 스펙의 dim)


In [ ]:
#@title 2. 코퍼스와 판정 세트
from embed import bakeoff, models
attractions = bakeoff.fetch_attractions(PLACE_API, lang=LANG, limit=DOC_LIMIT)
judgments = bakeoff.load_judgments(JUDGMENTS)
judged = [q for q in judgments if bakeoff.graded(q) and q.get("lang", "ko") == LANG]
print(f"docs={len(attractions)}  queries={len(judgments)}  judged({LANG})={len(judged)}")
if not judged:
    print("⚠ 판정된 질의가 없다 — judgments.yml 의 grade 를 먼저 채워야 nDCG 가 나온다. 상위 10 표만 나온다.")


In [ ]:
#@title 3. 모델별 평가 (한 모델씩 — 메모리를 비우며)
import gc, pandas as pd, torch
rows, per_query = [], {}
for key in MODELS:
    spec = models.resolve_revision(models.CANDIDATES[key])
    print(f"\n== {spec.ref}")
    try:
        model = bakeoff.load_model(spec, device="cuda")
        r, pq = bakeoff.evaluate(spec, model, attractions, judgments, dims=DIMS, rules=RULES, lang_filter=LANG)
        rows += r; per_query.update(pq)
        display(pd.DataFrame(r))
    except Exception as e:
        print("FAILED:", type(e).__name__, str(e)[:300])
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()
results = pd.DataFrame(rows + [bakeoff.bm25_baseline(judged)])
results


In [ ]:
#@title 4. 질의별 상위 10 — 판정 안 된 질의는 이 표를 보고 judgments.yml 의 vector_candidates 에 옮겨 판정한다
import pandas as pd
key, rule, dim = MODELS[0], "full", DIMS[0]
for q in judgments:
    if q.get("lang", "ko") != LANG: continue
    hits = per_query.get((key, rule, dim, q["query"]))
    if hits:
        print(f"\n### {q['query']}  ({q.get('intent','')})")
        display(pd.DataFrame(hits))


In [ ]:
#@title 4b. 영어 질의 — en 코퍼스(15k)로 같은 절차
attractions_en = bakeoff.fetch_attractions(PLACE_API, lang="en", limit=DOC_LIMIT)
rows_en = []
for key in MODELS:
    spec = models.resolve_revision(models.CANDIDATES[key])
    try:
        model = bakeoff.load_model(spec, device="cuda")
        r, _ = bakeoff.evaluate(spec, model, attractions_en, judgments, dims=DIMS, rules=["full"], lang_filter="en")
        rows_en += r
    except Exception as e:
        print(key, "FAILED:", str(e)[:200])
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(rows_en)


In [ ]:
#@title 5. 첫 채움 parquet (선택) — 고른 모델의 문서 벡터를 내린다. 업서트는 로컬 push --file 이 한다
if EXPORT_MODEL:
    spec = models.resolve_revision(models.CANDIDATES[EXPORT_MODEL])
    model = bakeoff.load_model(spec, device="cuda")
    corpus = bakeoff.build_corpus(attractions, "full", spec.ref)
    vecs = bakeoff.encode(model, corpus.texts, prompt=spec.doc_prompt, dim=spec.dim)
    out = f"attraction_embeddings_{LANG}_{spec.key}_d{spec.dim}.parquet"
    bakeoff.export_vectors(out, spec, corpus, vecs)
    print("saved", out, vecs.shape)


## 읽는 법

- `ndcg@10` 은 **판정된 질의만** 평균한다(`judged_queries`). 30 개 중 몇 개가 판정됐는지 같이 본다.
- 같은 모델에서 `dim 512` 가 `1024` 보다 0.01 안쪽으로 낮으면 512 로 간다 — k-NN 메모리가 반이다.
- `rule=title` 이 `full` 과 비슷하면 개요를 안 넣어도 된다는 뜻이고, 크게 낮으면 개요가 리콜을 만든다는 뜻이다.
- 마지막 행 `bm25 (운영)` 가 지금 검색이다. 이보다 못한 모델은 하이브리드로도 이득이 없다.
- 결과 표는 플랜 `docs/plans/2026-09-05-unified-search-hybrid-embedding.md` §8.3 옆에 붙인다.
